# Darija broadcast → .srt with LoRA + speaker labels (Kaggle T4)

Streamlined pipeline using the **anaszil LoRA adapter** on `large-v3-turbo`
with WhisperX word alignment and pyannote speaker diarization.

**Setup:**
- Settings → Accelerator → **GPU T4 x2** (or P100)
- Internet **On** (first run downloads models)
- For speaker labels: a HuggingFace read token (cell 3)

**Output:** `/kaggle/working/<clip>_darija.srt` with `[SPEAKER_XX]` prefixes.

## 1. Install dependencies

In [ ]:
!pip -q install "faster-whisper>=1.1.0" \
    "whisperx>=3.8.0" \
    "pyannote.audio>=3.1.0" \
    "transformers>=4.46.0" \
    "peft>=0.14.0"

## 2. Clone the repo & load pipeline code

Auto-clones the transcription repo to get the latest code and a sample clip.

In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/kaggle/working/haca-transcription")
if not REPO.exists():
    !git clone --depth 1 https://github.com/MarTCM/haca-transcription.git "{REPO}"
else:
    print("[repo] already cloned")

sys.path.insert(0, str(REPO / "src"))
import transcribe_whisperx as wx
from srt_writer import write_srt
print("[repo]", REPO)

## 3. HuggingFace token (REQUIRED for speaker labels)

Speaker diarization uses **pyannote.audio** gated models. You need:
1. A HuggingFace account at [huggingface.co](https://huggingface.co)
2. Accept usage terms at these two URLs (log in first):
   - https://huggingface.co/pyannote/speaker-diarization-community-1
   - https://huggingface.co/pyannote/segmentation-3.0
3. A **read token** from https://huggingface.co/settings/tokens

Paste the token below. **Without a token, pyannote is skipped** and you get
plain transcription (no `[SPEAKER_XX]` labels).

In [ ]:
HF_TOKEN = ""  # <-- paste your HuggingFace read token here

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN set")
else:
    print("WARNING: HF_TOKEN empty - diarization will be SKIPPED.")
    print("Speaker labels require a valid token (see instructions above).")

## 4. Choose a clip

Default: the trimmed broadcast sample from the repo. To use your own, upload
via **Add Data** → Upload and set `CLIP` to its path.

In [ ]:
import glob

CLIP = str(REPO / "samples" / "mobachara_ma3akom_trimmed.mp3")

uploaded = sorted(
    p for p in glob.glob("/kaggle/input/**/*", recursive=True)
    if os.path.splitext(p)[1].lower() in wx.MEDIA_EXTS
)
if uploaded:
    print("uploaded media files:")
    for p in uploaded:
        print("  ", p)

assert os.path.exists(CLIP), f"clip not found: {CLIP}"
print("\nusing CLIP:", CLIP)

## 5. Load model + LoRA adapter

Loads `openai/whisper-large-v3-turbo` in float16 via WhisperX, then loads the
`anaszil/whisper-large-v3-turbo-darija` LoRA adapter on top. Arabic chunks
will be routed through the LoRA; all other languages use CTranslate2.

First run downloads both models (~3 GB total).

In [ ]:
model = wx.load_model("large-v3-turbo", device="cuda", compute_type="float16")
lora_pipe = wx._load_darija_lora(device="cuda")
print("[done] model + LoRA loaded")

## 6. Transcribe with LoRA + speaker labels

Arabic chunks → LoRA adapter → word alignment → pyannote diarization.

The `words` field in LoRA segments lets WhisperX assign speakers even to
Arabic-transcribed text.

In [ ]:
segments = wx.transcribe_file(
    CLIP, model,
    lang="auto", allowed=("ar", "fr", "en"),
    max_chunk_s=25.0, batch_size=8, beam_size=5,
    diarize=True, hf_token=HF_TOKEN,
    device="cuda",
    darija_lora=True, lora_pipe=lora_pipe,
)

print(f"\n{len(segments)} cues total")

## 7. Check: did we get speaker labels?

In [ ]:
import re
spk = set()
for s in segments:
    m = re.match(r"^\[(SPEAKER_\d+)\]", s["text"])
    if m:
        spk.add(m.group(1))

if spk:
    print(f"{len(spk)} speaker(s) detected: {sorted(spk)}")
else:
    print("No speaker labels in output.")
    if not HF_TOKEN:
        print("Cause: HF_TOKEN was empty - diarization skipped.")
        print("Fix: set a valid token in cell 3 and re-run.")
    else:
        print("Token was set but diarization produced no labels.")
        print("Did you accept the pyannote model terms at the URLs in cell 3?")

## 8. Preview first cues

In [ ]:
for s in segments[:12]:
    print(f"  [{s['start']:6.1f}-{s['end']:6.1f}] {s['text']}")

from collections import Counter
langs = Counter()
for s in segments:
    lang = s.get("lang", "?")
    langs[lang] += 1
print(f"\nLanguage mix: {dict(langs)}")

## 9. Write the .srt

In [ ]:
out_name = os.path.splitext(os.path.basename(CLIP))[0]
out_path = write_srt(segments, f"/kaggle/working/{out_name}_darija.srt")
print("wrote", out_path)
print(out_path.read_text(encoding="utf-8")[:2000])

## 10. Verify the .srt is valid

In [ ]:
import re
BLOCK = re.compile(
    r"(\d+)\n"
    r"(\d\d:\d\d:\d\d,\d\d\d) --> (\d\d:\d\d:\d\d,\d\d\d)\n"
    r"(.+?)(?=\n\n|\Z)", re.DOTALL
)
text = out_path.read_text(encoding="utf-8")
cues = BLOCK.findall(text)
print(f"parsed {len(cues)} cues; first:", cues[0] if cues else None)
assert len(cues) == len(segments), f"cue count mismatch: {len(cues)} vs {len(segments)}"
print("SRT format OK - download from Kaggle Output panel")

---
## Summary

| Setting | Value |
|---------|-------|
| Model | `large-v3-turbo` + `anaszil/whisper-large-v3-turbo-darija` LoRA |
| Pipeline | WhisperX (alignment + diarization) |
| Language detection | Per-chunk, allow-list `ar,fr,en` |
| Arabic chunks | Routed through LoRA adapter, then aligned + diarized |
| Output | `/kaggle/working/<clip>_darija.srt` |

Download the `.srt` from the Kaggle **Output** panel (`/kaggle/working/`).
Each cue is prefixed with `[SPEAKER_00]` when diarization succeeds.